# バーレカンプ・マッシー（BM）法によるエラー位置多項式の生成

<a href="https://colab.research.google.com/github/Tsukumo-999/qr-code-from-scratch/blob/master/Step3/2_berlekamp_massey.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

第9回記事で行った、リードソロモン符号のシンドロームから「エラー位置多項式」を導出するBM法の実験ノートブックです。

### 1. サンプルの準備と2箇所の誤り混入
前回と同じガロア体の環境を準備し、「HAL」のデータに対してわざと **2箇所** のエラーを混入させてシンドロームを計算します。

In [1]:
# ガロア体 GF(2^8) の準備
exp_table = [0] * 256
log_table = [0] * 256
v = 1
for i in range(255):
    exp_table[i] = v
    log_table[v] = i
    v <<= 1
    if v & 0x100: v ^= 0x11D
exp_table[255] = exp_table[0]

def gf_mul(x, y):
    if x == 0 or y == 0: return 0
    return exp_table[(log_table[x] + log_table[y]) % 255]

def gf_div(x, y):
    if y == 0: raise ZeroDivisionError("0除算")
    if x == 0: return 0
    return exp_table[(log_table[x] - log_table[y]) % 255]

# 正しいメッセージ
correct_msg = [72, 65, 76, 97, 162, 112, 246]

# わざと2箇所のエラーを起こす
received_message = correct_msg.copy()
received_message[1] = 99  # 65 を 99 に破壊 (後ろから6番目)
received_message[3] = 200 # 97 を 200 に破壊 (後ろから4番目)

print(f"完璧なデータ: {correct_msg}")
print(f"受信データ　: {received_message}")

# シンドロームの計算
num_ec_codewords = 4
syndromes = []
for i in range(num_ec_codewords):
    alpha_i = exp_table[i]
    syndrome = 0
    for coef in received_message:
        syndrome = coef ^ gf_mul(syndrome, alpha_i)
    syndromes.append(syndrome)

print("\n▼ 計算されたシンドローム ▼")
for i, s in enumerate(syndromes):
    print(f" S_{i} = {s}")

完璧なデータ: [72, 65, 76, 97, 162, 112, 246]
受信データ　: [72, 99, 76, 200, 162, 112, 246]

▼ 計算されたシンドローム ▼
 S_0 = 139
 S_1 = 21
 S_2 = 219
 S_3 = 80


### 2. バーレカンプ・マッシー（BM）法の実装
BM法は、過去のシンドロームから次を予測し、外れたら（ディスクレパンシー $d \neq 0$）、多項式を修正していくアルゴリズムです。

* **C(x)** : 現在育てているエラー位置多項式
* **B(x)** : 最後に予測を外したときの古い多項式（これを使って修正を行う）
* **L** : 現在想定しているエラーの数
* **d** : ディスクレパンシー（ズレの大きさ）

In [ ]:
print("【BM法によるエラー位置多項式の生成開始】\n")

C = [1]  # 現在のエラー位置多項式 C(x)
B = [1]  # 古い多項式 B(x)
L = 0    # エラーの数
m = 1    # シフト量
b = 1    # 過去のディスクレパンシー

for i in range(len(syndromes)):
    # 1. ディスクレパンシー (d) の計算（予測と実際のズレ）
    d = syndromes[i]
    for j in range(1, L + 1):
        d ^= gf_mul(C[j], syndromes[i - j])
        
    print(f"=== ステップ {i} ===")
    print(f"  ディスクレパンシー d = {d}")
    
    # 2. ズレがない場合（予測成功）
    if d == 0:
        m += 1
        print(f"  ✅ 予測成功: 多項式に変更なし C(x) = {C}")
        
    # 3. ズレがある場合（予測失敗 -> 多項式のアップデート）
    else:
        # 修正項の計算: (d / b) * B(x) * x^m
        coef = gf_div(d, b)
        term = [0] * m + [gf_mul(coef, c) for c in B]
        
        # 次のC(x)を作成（C(x) と 修正項を XOR）
        next_C = [0] * max(len(C), len(term))
        for k in range(len(next_C)):
            val_C = C[k] if k < len(C) else 0
            val_term = term[k] if k < len(term) else 0
            next_C[k] = val_C ^ val_term
            
        # 古い多項式 B(x) を更新すべきかの判定
        if 2 * L <= i:
            L = i + 1 - L
            B = C.copy()
            b = d
            m = 1
            print(f"  予測失敗(ベース更新): エラー数 L が {L} に増加。 新しい C(x) = {next_C}")
        else:
            m += 1
            print(f"  予測失敗(ベース維持): 新しい C(x) = {next_C}")
            
        C = next_C

print("\n====================================")
print(f" 最終的なエラー位置多項式 σ(x): {C}")
print("====================================")
print(f"(配列のインデックスが x の次数に対応します: {C[0]} + {C[1]}x + {C[2]}x^2)")

【BM法によるエラー位置多項式の生成開始】

=== ステップ 0 ===
  ディスクレパンシー d = 139
  ❌ 予測失敗(ベース更新): エラー数 L が 1 に増加。 新しい C(x) = [1, 139]
=== ステップ 1 ===
  ディスクレパンシー d = 67
  ❌ 予測失敗(ベース維持): 新しい C(x) = [1, 115]
=== ステップ 2 ===
  ディスクレパンシー d = 26
  ❌ 予測失敗(ベース更新): エラー数 L が 2 に増加。 新しい C(x) = [1, 115, 197]
=== ステップ 3 ===
  ディスクレパンシー d = 141
  ❌ 予測失敗(ベース維持): 新しい C(x) = [1, 40, 29]

🎯 最終的なエラー位置多項式 σ(x): [1, 40, 29]
(配列のインデックスが x の次数に対応します: 1 + 40x + 29x^2)


### 3. ここからはチェン探索
方程式（エラー位置多項式）が完成しました！
あとは、ガロア体の要素 $\alpha^0 \sim \alpha^6$ をこの方程式の $x$ に順番に代入していき、計算結果が `0` になる要素（根）を見つけるだけです。

それがエラーの正確な「場所」を教えてくれます。次回「チェン探索（Chien Search）」に続きます。

[QRコードのデコード 誤り位置を特定するチェン探索（Chien Search）【QRコードを解読する 第10回】](https://tukumolog.com/qr-code-decode-10-chien-search/)

## 4. 参考資料とリンク
* **[QRコードを読み解く：シンドロームの計算とエラー検出【QRコードを解読する 第8回】](https://tukumolog.com/qr-code-decode-08-syndrome-calculation/)**
* **[ガロア体入門 (Theoretical Background)](https://theoretical-background.com/%E3%82%AC%E3%83%AD%E3%82%A2%E4%BD%93%E5%85%A5%E9%96%80/)**